In [2]:
import pandas as pd
import whisper
import os
from jiwer import wer, cer
from tqdm import tqdm
import torch


In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [11]:
device

'cuda'

In [ ]:

# 1. 초기 설정
device = "cuda" if torch.cuda.is_available() else "cpu"
model = whisper.load_model("medium").to(device) # Whisper 모델 로드
gt_df = pd.read_csv('/Dataset/fine-tuning_data.csv') # GT 파일 로드 (파일명: file, 텍스트: text)
audio_dir = '/Dataset/voice/'  # 음성 파일 경로

# 2. 결과 저장을 위한 리스트
stt_results = []

print("Starting STT Inference and EDA...")
count = 0
for idx, row in tqdm(gt_df.iterrows(), total=len(gt_df)):
    audio_path = row['path'] # Sub01_a1
    gt_text = row['GT-STT']
    emotion = row['emotion']

    if not os.path.exists(audio_path):
        continue

    # Whisper 추론 (48kHz -> 16kHz 리샘플링 자동 처리)
    result = model.transcribe(audio_path, language="ko")
    pred_text = result['text'].strip()

    # CER, WER 계산
    error_wer = wer(gt_text, pred_text)
    error_cer = cer(gt_text, pred_text)

    # 데이터 저장
    stt_results.append({
        'file_id': audio_path,
        'subject': audio_path.split('/')[-1],
        'emotion': emotion,
        'gt_text': gt_text,
        'pred_text': pred_text,
        'wer': error_wer,
        'cer': error_cer
    })

# 3. 데이터프레임 변환 및 확인
analysis_df = pd.DataFrame(stt_results)
print(analysis_df[['file_id', 'wer', 'cer']].head())

new_emotion_to_senior_idx = {
    'happiness': 0,
    'neutral': 1,
    'fear': 2,
    'disgust': 3,
    'surprise': 4,
    'sad': 5,
    'angry': 6,
}

In [6]:
# 1. 초기 설정
# Whisper 모델 로드
gt_df = pd.read_csv('/Dataset/fine-tuning_data.csv') # GT 파일 로드 (파일명: file, 텍스트: text)
audio_dir = '/Dataset/voice/'  # 음성 파일 경로

# 2. 결과 저장을 위한 리스트
stt_results = []

count = 0
for idx, row in tqdm(gt_df.iterrows(), total=len(gt_df)):
    audio_path = row['path'] # Sub01_a1
    gt_text = row['GT-STT']
    emotion = row['emotion']


    # 데이터 저장
    stt_results.append({
        'emotion': emotion,
        'gt_text': gt_text,
    })

# 3. 데이터프레임 변환 및 확인
analysis_df = pd.DataFrame(stt_results)

train_df = analysis_df[['gt_text', 'emotion']]
train_df.columns = ['input_text', 'emotion']

train_df.to_csv("./target_kobert_train_data.tsv", sep='\t', index=False)

In [4]:
# 감정별 평균 CER 확인 (EDA

# KoBART 학습용 데이터셋 생성 (CER 30% 이하인 데이터만 권장)
# Input: STT 결과(오타 포함 가능), Target: 실제 GT 텍스트
train_df = analysis_df[['gt_text', 'emotion']]
train_df.columns = ['target_text', 'emotion']

train_df.to_csv("./target_kobert_train_data.tsv", sep='\t', index=False)

In [7]:
train_df

,input_text,target_text,emotion
0,아 청소 니가 대신 해줘,"어, 청소 니가 대신 해 줘!",angry
1,둘 다 청소하기 싫어 귀찮아,둘 다 청소 하기 싫어. 귀찮아.,angry
2,둘 다 하기 싫어서 화내,둘 다 하기 싫어서 화내.,angry
5,그냥 걷고 있어,그냥 걷고 있어.,sad
6,어 고등학교 동창인데 아 이렇게 더럽게 쓸 줄 몰랐어,어. 고등학교 동창인데 이렇게 더럽게 쓸줄 몰랐어.,angry
7,처음 학원에서 만났다가 서로 좋아해서 사귀게 되었지,처음 학원에서 만났다가 서로 좋아해서 사귀게 되었지.,sad
8,내가 애정표현을 잘 못해서 자주 싸우긴 했어,내가 애정 표현을 잘 못해서 자주 싸우긴 했어.,sad
9,오늘 헤어졌어,오늘 헤어졌어.,sad
